In this notebook we will take a closer look into the Random Forest algorithm. First lets do a Random Forest without any hyperparameter tuning and then we will do a Random Forest with hyperparameter tuning.


In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import SelectKBest, f_classif

In [22]:

# Load the dataset
df = pd.read_csv('winequality-red-clean.csv')
X = df.drop('quality', axis=1)  # Features
y = df['quality']  # Target (quality)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [23]:
# Feature Selection: Select Top 5 Features Using SelectKBest
k = 5  # Number of top features to select
selector = SelectKBest(score_func=f_classif, k=k)

# Fit to training data
X_train = selector.fit_transform(X_train, y_train)
X_test = selector.transform(X_test)  # Apply same transformation to test data

# Print selected feature names
selected_features = selector.get_support(indices=True)
print(f"Selected top {k} features: {X.columns[selected_features]}")


Selected top 5 features: Index(['volatile acidity', 'citric acid', 'total sulfur dioxide', 'sulphates',
       'alcohol'],
      dtype='object')


In [24]:

# Initialize a DataFrame to store performance results
performance_df = pd.DataFrame(columns=[
    "Model",
    "Train Accuracy",
    "Test Accuracy",
    "Cross-Validation Accuracy (Mean ± Std)",
    "Bias",
    "Variance"
])

# 1. Random Forest without Hyperparameter Tuning
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# Predict on training and test data
y_train_pred = rf_model.predict(X_train)
y_test_pred = rf_model.predict(X_test)

# Calculate accuracy for both train and test sets
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

# Cross-validation results (5-fold)
cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5)
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

# Calculate Bias and Variance
bias = 100 - (train_accuracy * 100)  # Bias as 100 - train accuracy
variance = (train_accuracy - test_accuracy) * 100  # Variance as the difference between train and test accuracy

# Store the results in the performance_df DataFrame
performance_df.loc[len(performance_df.index)] = [
    "Random Forest (No Tuning)",
    f"{train_accuracy * 100:.2f}%",
    f"{test_accuracy * 100:.2f}%",
    f"{cv_mean * 100:.2f}% ± {cv_std * 100:.2f}%",
    f"{bias:.2f}%",
    f"{variance:.2f}%"
]

performance_df


,Model,Train Accuracy,Test Accuracy,Cross-Validation Accuracy (Mean ± Std),Bias,Variance
0,Random Forest (No Tuning),100.00%,69.52%,65.95% ± 2.36%,0.00%,30.48%


In [34]:

# 2. Random Forest with Hyperparameter Tuning
# 2. Random Forest with Hyperparameter Tuning
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2'],  # Added max_features parameter
    'oob_score': [True]  # Enable Out-of-Bag scoring
}

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state=42),
                           param_grid=param_grid,
                           scoring='accuracy',
                           cv=5,
                           n_jobs=-1)

# Fit the model
grid_search.fit(X_train, y_train)
best_rf_model = grid_search.best_estimator_

best_hyperparams = grid_search.best_params_
print("Best Hyperparameters:", best_hyperparams)

# Predict on training and test data
y_train_pred_tuned = best_rf_model.predict(X_train)
y_test_pred_tuned = best_rf_model.predict(X_test)

# Calculate accuracy for both train and test sets
train_accuracy_tuned = accuracy_score(y_train, y_train_pred_tuned)
test_accuracy_tuned = accuracy_score(y_test, y_test_pred_tuned)

# Cross-validation results (5-fold)
cv_scores_tuned = cross_val_score(best_rf_model, X_train, y_train, cv=5)
cv_mean_tuned = cv_scores_tuned.mean()
cv_std_tuned = cv_scores_tuned.std()

# Calculate Bias and Variance
bias_tuned = 100 - (train_accuracy_tuned * 100)  # Bias as 100 - train accuracy
variance_tuned = (
                         train_accuracy_tuned - test_accuracy_tuned) * 100  # Variance as the difference between train and test accuracy

# Store the results in the performance_df DataFrame
performance_df.loc[len(performance_df.index)] = [
    "Random Forest (With Tuning)",
    f"{train_accuracy_tuned * 100:.2f}%",
    f"{test_accuracy_tuned * 100:.2f}%",
    f"{cv_mean_tuned * 100:.2f}% ± {cv_std_tuned * 100:.2f}%",
    f"{bias_tuned:.2f}%",
    f"{variance_tuned:.2f}%"
]

print(f"Random Forest (With Tuning) OOB:  {best_rf_model.oob_score_}")
# Display the results table
performance_df

Best Hyperparameters: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_split': 5, 'n_estimators': 200, 'oob_score': True}


TypeError: can only concatenate str (not "float") to str

AttributeError: 'GridSearchCV' object has no attribute 'best_params_'

Best Hyperparameters: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_split': 5, 'n_estimators': 200, 'oob_score': True}


Here we can see that the Random Forest model with hyperparameter has a lower test accuracy than the Random Forest model without hyperparameter tuning, but the cross-validation accuracy is higher, which is more important.The variance is also lower for the model with hyperparameter tuning. The oob score is extremely close to the test accuracy which is a good sign. 
The variance is still high, so we can try to reduce it by using regularization.

In [28]:
# 3. Random Forest with Hyperparameter Tuning and Regularization
#  Set up the parameter grid for tuning with regularization
param_grid = {
    'n_estimators': [50, 100, 200],  # Number of trees in the forest
    'max_depth': [None, 10, 20, 30],  # Maximum depth of the tree
    'min_samples_split': [5, 10, 12],  # Minimum number of samples required to split an internal node
    'max_features': ['sqrt', 'log2'],  # Number of features to consider for the best split
    'min_samples_leaf': [2, 3, 4],  # Minimum number of samples required to be at a leaf node
    'oob_score': [True]  # Enable Out-of-Bag scoring

}

# Initialize GridSearchCV
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5)

# Fit the model with hyperparameter tuning
grid_search.fit(X_train, y_train)

# Get the best model from grid search
best_rf_model = grid_search.best_estimator_

# Predict on training data and test data with the best model
y_train_pred_tuning = best_rf_model.predict(X_train)
y_test_pred_tuning = best_rf_model.predict(X_test)

# Accuracy for both train and test sets
train_accuracy_tuning = accuracy_score(y_train, y_train_pred_tuning)
test_accuracy_tuning = accuracy_score(y_test, y_test_pred_tuning)

# Cross-validation results (5-fold) with the best model
cv_scores_tuning = cross_val_score(best_rf_model, X_train, y_train, cv=5)
cv_mean_tuning = cv_scores_tuning.mean()
cv_std_tuning = cv_scores_tuning.std()

# Calculate Bias and Variance
bias_tuning = 100 - (train_accuracy_tuning * 100)  # Bias as 100 - train accuracy
variance_tuning = (train_accuracy_tuning - test_accuracy_tuning) * 100  # Variance

# Store the results in the performance_df DataFrame
performance_df.loc[len(performance_df.index)] = [
    "Random Forest (With Tuning and Regularization)",
    f"{train_accuracy_tuning * 100:.2f}%",
    f"{test_accuracy_tuning * 100:.2f}%",
    f"{cv_mean_tuning * 100:.2f}% ± {cv_std_tuning * 100:.2f}%",
    f"{bias_tuning:.2f}%",
    f"{variance_tuning:.2f}%"
]
print(f"Random Forest (With Tuning and Regularization) OOB: {best_rf_model.oob_score_}")

# Display the performance results
performance_df

Random Forest (With Tuning and Regularization) OOB: 0.6723842195540308


,Model,Train Accuracy,Test Accuracy,Cross-Validation Accuracy (Mean ± Std),Bias,Variance
0,Random Forest (No Tuning),100.00%,69.52%,65.95% ± 2.36%,0.00%,30.48%
1,Random Forest (With Tuning),97.77%,68.84%,66.64% ± 2.78%,2.23%,28.93%
2,Random Forest (With Tuning and Regularization),93.05%,68.15%,65.70% ± 2.47%,6.95%,24.90%
3,Random Forest (With Tuning and Regularization),93.05%,68.15%,65.70% ± 2.47%,6.95%,24.90%
4,Random Forest (With Tuning and Regularization),93.05%,68.15%,65.70% ± 2.47%,6.95%,24.90%


What we did: Increasing `min_samples_leaf` and ``min_samples_split`` to encourage broader, more generalized splits.

Results: all accuracy metrics and variance have decreased. The OOB score is also very close to the test accuracy, which is a good sign. But this begs the question is it worth is bringing the variance down at the cost of accuracy? In some real-world scenarios, it might be worth it, but in others, it might not be. In our case this model is not life-threatening, so we can afford to have a higher variance. Also, we have chosen to have accuracy as our main metric, so this model is not the best for us.


### How does OOB Error work?
Bootstrap Sampling:
 When training a Random Forest, multiple decision trees are created using different bootstrap samples from the original dataset. A bootstrap sample is generated by randomly sampling with replacement from the dataset.

OOB Samples: Since the sample is drawn with replacement, about one-third of the data points are left out of each bootstrap sample. These points are the OOB samples for that particular tree.

Prediction Using OOB Samples: When making predictions, the OOB samples can be fed into the trees that did not use them during training. This allows you to assess the performance of the model on data it hasn’t seen.

Error Estimation: By aggregating the predictions from all the trees that did not use a particular observation, you can compute an OOB prediction for that observation. The OOB error is then calculated based on the discrepancies between the predicted and actual values across all observations.

Lets try to use AdaBoost with Random Forest as the base estimator to see if we can improve the accuracy.

In [31]:
# Import AdaBoost Classifier
from sklearn.ensemble import AdaBoostClassifier

# Initialize a Random Forest model as the base estimator
base_rf_model = RandomForestClassifier(n_estimators=50, random_state=42)

# Initialize the AdaBoostClassifier with the RandomForestClassifier as the base estimator
adaboost_rf_model = AdaBoostClassifier(
    estimator=base_rf_model,
    n_estimators=50,  # Number of boosting rounds
    learning_rate=1.0,  # Weight applied to each classifier
    random_state=42,
    algorithm='SAMME'  # Discrete AdaBoost as samme.r is deprecated
)

# Fit the AdaBoost model with the RandomForest base estimator
adaboost_rf_model.fit(X_train, y_train)

# Predict on training and test data
y_train_pred_ada = adaboost_rf_model.predict(X_train)
y_test_pred_ada = adaboost_rf_model.predict(X_test)

# Calculate accuracy for both train and test sets
train_accuracy_ada = accuracy_score(y_train, y_train_pred_ada)
test_accuracy_ada = accuracy_score(y_test, y_test_pred_ada)

# Cross-validation results (5-fold)
cv_scores_ada = cross_val_score(adaboost_rf_model, X_train, y_train, cv=5)
cv_mean_ada = cv_scores_ada.mean()
cv_std_ada = cv_scores_ada.std()

# Calculate Bias and Variance
bias_ada = 100 - (train_accuracy_ada * 100)  # Bias as 100 - train accuracy
variance_ada = (
                       train_accuracy_ada - test_accuracy_ada) * 100  # Variance as the difference between train and test accuracy

# Store the results in the performance_df DataFrame
performance_df.loc[len(performance_df.index)] = [
    "Random Forest with AdaBoost",
    f"{train_accuracy_ada * 100:.2f}%",
    f"{test_accuracy_ada * 100:.2f}%",
    f"{cv_mean_ada * 100:.2f}% ± {cv_std_ada * 100:.2f}%",
    f"{bias_ada:.2f}%",
    f"{variance_ada:.2f}%"
]

# Display the performance results
performance_df


,Model,Train Accuracy,Test Accuracy,Cross-Validation Accuracy (Mean ± Std),Bias,Variance
0,Random Forest (No Tuning),100.00%,69.52%,65.95% ± 2.36%,0.00%,30.48%
1,Random Forest (With Tuning),97.77%,68.84%,66.64% ± 2.78%,2.23%,28.93%
2,Random Forest (With Tuning and Regularization),93.05%,68.15%,65.70% ± 2.47%,6.95%,24.90%
3,Random Forest with AdaBoost,100.00%,67.81%,65.78% ± 2.33%,0.00%,32.19%


Lowest test accuracy so far and not the best cross-validation accuracy. The variance is also high. And a 100% train accuracy. Let's try to tune the hyperparameters to see if we can improve the model.

In [35]:
# Hyperparameter tuning for AdaBoost with Random Forest
# Import necessary libraries
import pandas as pd
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.metrics import accuracy_score

# Sample dataset creation (replace this with your actual dataset)
# X, y = ... # Your feature matrix and target variable

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize a Random Forest model as the base estimator
base_rf_model = RandomForestClassifier(n_estimators=50, random_state=42)

# Initialize the AdaBoostClassifier with the RandomForestClassifier as the base estimator
adaboost_model = AdaBoostClassifier(estimator=base_rf_model)

# Define hyperparameter grid for tuning
param_grid = {
    'n_estimators': [50, 100, 200],  # Number of boosting rounds
    'learning_rate': [0.01, 0.1, 1.0],  # Weight applied to each classifier
    'algorithm': ['SAMME', 'SAMME.R']  # Boosting algorithms
}

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=adaboost_model,
                           param_grid=param_grid,
                           scoring='accuracy',
                           cv=5,
                           n_jobs=-1)

# Fit the model with hyperparameter tuning
grid_search.fit(X_train, y_train)

# Get the best model and its parameters
best_ada_model = grid_search.best_estimator_
best_params = grid_search.best_params_

# Predict on training data and test data with the best model
y_train_pred_ada = best_ada_model.predict(X_train)
y_test_pred_ada = best_ada_model.predict(X_test)

# Calculate accuracy for both train and test sets
train_accuracy_ada = accuracy_score(y_train, y_train_pred_ada)
test_accuracy_ada = accuracy_score(y_test, y_test_pred_ada)

# Cross-validation results (5-fold)
cv_scores_ada = cross_val_score(best_ada_model, X_train, y_train, cv=5)
cv_mean_ada = cv_scores_ada.mean()
cv_std_ada = cv_scores_ada.std()

# Calculate Bias and Variance
bias_ada = 100 - (train_accuracy_ada * 100)  # Bias as 100 - train accuracy
variance_ada = (train_accuracy_ada - test_accuracy_ada) * 100  # Variance

# Store the results in the performance_df DataFrame
performance_df.loc[len(performance_df.index)] = [
    "Random Forest with AdaBoost (Tuned)",
    f"{train_accuracy_ada * 100:.2f}%",
    f"{test_accuracy_ada * 100:.2f}%",
    f"{cv_mean_ada * 100:.2f}% ± {cv_std_ada * 100:.2f}%",
    f"{bias_ada:.2f}%",
    f"{variance_ada:.2f}%"
]

# Display the performance results
print("Best Hyperparameters for AdaBoost:", best_params)
performance_df


C:\Users\margi\PycharmProjects\data_5\.venv\lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
C:\Users\margi\PycharmProjects\data_5\.venv\lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
C:\Users\margi\PycharmProjects\data_5\.venv\lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
C:\Users\margi\PycharmProjects\data_5\.venv\lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use t

Best Hyperparameters for AdaBoost: {'algorithm': 'SAMME.R', 'learning_rate': 0.01, 'n_estimators': 50}


C:\Users\margi\PycharmProjects\data_5\.venv\lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


,Model,Train Accuracy,Test Accuracy,Cross-Validation Accuracy (Mean ± Std),Bias,Variance
0,Random Forest (No Tuning),100.00%,69.52%,65.95% ± 2.36%,0.00%,30.48%
1,Random Forest (With Tuning),97.77%,68.84%,66.64% ± 2.78%,2.23%,28.93%
2,Random Forest (With Tuning and Regularization),93.05%,68.15%,65.70% ± 2.47%,6.95%,24.90%
3,Random Forest (With Tuning),97.77%,68.84%,66.64% ± 2.78%,2.23%,28.93%
4,Random Forest with AdaBoost (Tuned),100.00%,70.55%,68.27% ± 3.10%,0.00%,29.45%


We finally cracked the 70% test accuracy! The cross-validation accuracy is also higher than the previous models. The variance is also lower. This is looking good! 

Let's try to use Extra Trees to see if we can improve the accuracy even further.

In [36]:
from sklearn.ensemble import ExtraTreesClassifier

# 1. Extra Trees without Hyperparameter Tuning
extra_trees_model = ExtraTreesClassifier(random_state=42)
extra_trees_model.fit(X_train, y_train)

# Predict on training and test data
y_train_pred = extra_trees_model.predict(X_train)
y_test_pred = extra_trees_model.predict(X_test)

# Calculate accuracy for both train and test sets
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

# Cross-validation results (5-fold)
cv_scores = cross_val_score(extra_trees_model, X_train, y_train, cv=5)
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

# Calculate Bias and Variance
bias = 100 - (train_accuracy * 100)  # Bias as 100 - train accuracy
variance = (train_accuracy - test_accuracy) * 100  # Variance

# Store the results in the performance_df DataFrame
performance_df.loc[len(performance_df.index)] = [
    "Extra Trees (No Tuning)",
    f"{train_accuracy * 100:.2f}%",
    f"{test_accuracy * 100:.2f}%",
    f"{cv_mean * 100:.2f}% ± {cv_std * 100:.2f}%",
    f"{bias:.2f}%",
    f"{variance:.2f}%"
]

# Display the performance results
performance_df

,Model,Train Accuracy,Test Accuracy,Cross-Validation Accuracy (Mean ± Std),Bias,Variance
0,Random Forest (No Tuning),100.00%,69.52%,65.95% ± 2.36%,0.00%,30.48%
1,Random Forest (With Tuning),97.77%,68.84%,66.64% ± 2.78%,2.23%,28.93%
2,Random Forest (With Tuning and Regularization),93.05%,68.15%,65.70% ± 2.47%,6.95%,24.90%
3,Random Forest (With Tuning),97.77%,68.84%,66.64% ± 2.78%,2.23%,28.93%
4,Random Forest with AdaBoost (Tuned),100.00%,70.55%,68.27% ± 3.10%,0.00%,29.45%
5,Extra Trees (No Tuning),100.00%,72.60%,69.13% ± 0.69%,0.00%,27.40%


Extra tress have the best results so far even better than the tuned ada boost model. Let's try to tune the hyperparameters to see if we can improve the model even further.

In [40]:
# Hyperparameter tuning for Extra Trees Classifier
param_grid = {
    'n_estimators': [50, 100, 200, 250],  # Number of trees in the forest
    'max_depth': [None, 10, 20, 30],  # Maximum depth of the trees
    'min_samples_split': [2, 5, 10],  # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2, 4],  # Minimum number of samples required to be at a leaf node
    'max_features': ['sqrt', 'log2'],  # Number of features to consider when looking for the best split
    'bootstrap': [True, False]  # Add bootstrap option
}

# Initialize GridSearchCV
grid_search = GridSearchCV(ExtraTreesClassifier(random_state=42), param_grid, cv=5, n_jobs=-1, scoring='accuracy')

# Fit the model with hyperparameter tuning
grid_search.fit(X_train, y_train)

# Get the best model from grid search
best_extra_trees_model = grid_search.best_estimator_

# Predict on training and test data with the best model
y_train_pred_tuned = best_extra_trees_model.predict(X_train)
y_test_pred_tuned = best_extra_trees_model.predict(X_test)

# Calculate accuracy for both train and test sets
train_accuracy_tuned = accuracy_score(y_train, y_train_pred_tuned)
test_accuracy_tuned = accuracy_score(y_test, y_test_pred_tuned)

# Cross-validation results (5-fold) with the best model
cv_scores_tuned = cross_val_score(best_extra_trees_model, X_train, y_train, cv=5)
cv_mean_tuned = cv_scores_tuned.mean()
cv_std_tuned = cv_scores_tuned.std()

# Calculate Bias and Variance
bias_tuned = 100 - (train_accuracy_tuned * 100)  # Bias as 100 - train accuracy
variance_tuned = (train_accuracy_tuned - test_accuracy_tuned) * 100  # Variance

# Store the results in the performance_df DataFrame
performance_df.loc[len(performance_df.index)] = [
    "Extra Trees (With Tuning)",
    f"{train_accuracy_tuned * 100:.2f}%",
    f"{test_accuracy_tuned * 100:.2f}%",
    f"{cv_mean_tuned * 100:.2f}% ± {cv_std_tuned * 100:.2f}%",
    f"{bias_tuned:.2f}%",
    f"{variance_tuned:.2f}%"
]

# Display the performance results
performance_df

,Model,Train Accuracy,Test Accuracy,Cross-Validation Accuracy (Mean ± Std),Bias,Variance
0,Random Forest (No Tuning),100.00%,69.52%,65.95% ± 2.36%,0.00%,30.48%
1,Random Forest (With Tuning),97.77%,68.84%,66.64% ± 2.78%,2.23%,28.93%
2,Random Forest (With Tuning and Regularization),93.05%,68.15%,65.70% ± 2.47%,6.95%,24.90%
3,Random Forest (With Tuning),97.77%,68.84%,66.64% ± 2.78%,2.23%,28.93%
4,Random Forest with AdaBoost (Tuned),100.00%,70.55%,68.27% ± 3.10%,0.00%,29.45%
5,Extra Trees (No Tuning),100.00%,72.60%,69.13% ± 0.69%,0.00%,27.40%
6,Extra Trees (With Tuning),99.83%,69.18%,69.90% ± 2.58%,0.17%,30.65%
7,Extra Trees (With Tuning),99.83%,69.18%,69.90% ± 2.58%,0.17%,30.65%


Tuning didn't help much. Results are similar and slightly worse to the model without tuning. We will use the extra trees model without tuning as our final model that will be used in an ansamble model.

In [38]:
performance_df = performance_df.drop(performance_df.index[-1:])